# BigAlpha 2026 约束搜索

在AIStudio内生成48个候选。先用小样本核对字段，再切换4C/16G运行完整历史。

In [ ]:
"""Shared, submission-safe data and feature utilities.

This module deliberately uses only numpy, pandas and optional BigQuant ``dai``.
The submission notebook builder embeds this file into each standalone notebook.
"""


from dataclasses import dataclass
from typing import Iterable, Mapping, Sequence

import numpy as np
import pandas as pd

try:  # Available in BigQuant AIStudio, absent in local unit tests.
    import dai  # type: ignore
except Exception:  # pragma: no cover - exercised only on BigQuant.
    dai = None


EPS = 1e-12
BAR_ALIASES = ("bar1m", "stock_bar1m", "bigalpha_2026_stock_bar1m")
INSTRUMENT_ALIASES = ("instruments", "instrument", "bigalpha_2026_instruments")
FINANCIAL_ALIASES = ("financial", "financial_statement", "bigalpha_2026_financial")


@dataclass(frozen=True)
class HFFeatureConfig:
    """Parameters selected by the constrained search."""

    depth: int = 5
    tail_minutes: int = 60
    replenishment_weight: float = 0.30
    microprice_weight: float = 0.20


DEFAULT_HF_CONFIG = HFFeatureConfig()


def normalize_date(value: object) -> pd.Timestamp:
    return pd.Timestamp(value).normalize()


def _as_timestamp_bounds(start_date: object, end_date: object) -> tuple[pd.Timestamp, pd.Timestamp]:
    start = normalize_date(start_date)
    end = normalize_date(end_date)
    if end < start:
        raise ValueError("end_date must be on or after start_date")
    return start, end


def resolve_source(
    datasources: object,
    aliases: Sequence[str],
    default_table: str,
) -> object:
    """Resolve a BigQuant datasource mapping, dataframe, query object or table name."""

    if isinstance(datasources, Mapping):
        for name in aliases:
            value = datasources.get(name)
            if value is not None:
                return value
    return default_table


def _filter_frame(
    frame: pd.DataFrame,
    fields: Sequence[str],
    left: object,
    right_exclusive: object,
) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()
    result = frame.copy()
    if "date" in result.columns:
        result["date"] = pd.to_datetime(result["date"], errors="coerce")
        result = result[
            (result["date"] >= pd.Timestamp(left))
            & (result["date"] < pd.Timestamp(right_exclusive))
        ]
    if "instrument" in result.columns:
        result["instrument"] = result["instrument"].astype(str)
    available = [name for name in fields if name in result.columns]
    return result.loc[:, available].copy()


def query_table(
    datasources: object,
    aliases: Sequence[str],
    default_table: str,
    fields: Sequence[str],
    left: object,
    right_exclusive: object,
) -> pd.DataFrame:
    """Read a table with a local DataFrame and BigQuant-compatible fallback."""

    source = resolve_source(datasources, aliases, default_table)
    if isinstance(source, pd.DataFrame):
        return _filter_frame(source, fields, left, right_exclusive)

    filters = {
        "date": [
            pd.Timestamp(left).strftime("%Y-%m-%d %H:%M:%S"),
            pd.Timestamp(right_exclusive).strftime("%Y-%m-%d %H:%M:%S"),
        ]
    }
    select = "SELECT " + ", ".join(fields)
    if hasattr(source, "query") and not isinstance(source, str):
        try:
            frame = source.query(select, filters=filters).df()
            return _filter_frame(frame, fields, left, right_exclusive)
        except Exception:
            pass

    if dai is None:
        return pd.DataFrame(columns=list(fields))

    try:
        frame = dai.query(
            f"{select} FROM {source}",
            filters=filters,
            compression=True,
        ).df()
        return _filter_frame(frame, fields, left, right_exclusive)
    except Exception:
        return pd.DataFrame(columns=list(fields))


def load_universe(
    datasources: object,
    start_date: object,
    end_date: object,
) -> pd.DataFrame:
    start, end = _as_timestamp_bounds(start_date, end_date)
    frame = query_table(
        datasources,
        INSTRUMENT_ALIASES,
        "bigalpha_2026_instruments",
        ("date", "instrument"),
        start,
        end + pd.Timedelta(days=1),
    )
    if frame.empty or not {"date", "instrument"}.issubset(frame.columns):
        return pd.DataFrame(columns=["date", "instrument"])
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce").dt.normalize()
    frame["instrument"] = frame["instrument"].astype(str)
    return (
        frame.dropna(subset=["date", "instrument"])
        .drop_duplicates(["date", "instrument"])
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
    )


def _numeric(frame: pd.DataFrame, name: str, default: float = np.nan) -> pd.Series:
    if name not in frame.columns:
        return pd.Series(default, index=frame.index, dtype=float)
    return pd.to_numeric(frame[name], errors="coerce").replace([np.inf, -np.inf], np.nan)


def cumulative_to_increment(
    values: pd.Series,
    instruments: pd.Series,
    days: pd.Series,
) -> pd.Series:
    """Convert an intraday cumulative field to increments.

    The first observation of a day is its own increment. Negative differences
    are treated as exchange/vendor resets and restart from the current value.
    """

    numeric = pd.to_numeric(values, errors="coerce").fillna(0.0).clip(lower=0.0)
    keys = [instruments.astype(str), pd.to_datetime(days).dt.normalize()]
    diff = numeric.groupby(keys, sort=False).diff()
    first = numeric.groupby(keys, sort=False).cumcount().eq(0)
    increment = diff.where(~first, numeric)
    reset = increment.lt(0.0) | increment.isna()
    increment = increment.where(~reset, numeric)
    return increment.clip(lower=0.0)


def cross_section_rank(
    frame: pd.DataFrame,
    values: pd.Series,
    fill_neutral: bool = False,
) -> pd.Series:
    numeric = pd.to_numeric(values, errors="coerce").replace([np.inf, -np.inf], np.nan)
    ranked = numeric.groupby(frame["date"], sort=False).rank(pct=True, method="average")
    scaled = 2.0 * (ranked - 0.5)
    return scaled.fillna(0.0) if fill_neutral else scaled


def _safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    return (
        numerator.astype(float)
        .div(denominator.astype(float).where(denominator.abs() > EPS))
        .replace([np.inf, -np.inf], np.nan)
    )


def _book_expression(depth: int) -> tuple[str, str, str]:
    levels = range(1, depth + 1)
    bid = " + ".join(f"bid_volume{i} / {float(i):.1f}" for i in levels)
    ask = " + ".join(f"ask_volume{i} / {float(i):.1f}" for i in levels)
    imbalance = f"(({bid}) - ({ask})) / (ABS({bid}) + ABS({ask}) + 1e-12)"
    return bid, ask, imbalance


def _try_query_daily_hf_dai(
    datasources: object,
    start_date: object,
    end_date: object,
    config: HFFeatureConfig,
) -> pd.DataFrame:
    """Aggregate on the DAI server to avoid transferring minute snapshots."""

    if dai is None:
        return pd.DataFrame()
    source = resolve_source(
        datasources,
        BAR_ALIASES,
        "bigalpha_2026_stock_bar1m",
    )
    if not isinstance(source, str):
        return pd.DataFrame()

    bid, ask, imbalance = _book_expression(config.depth)
    threshold_by_window = {
        15: 144500000,
        30: 143000000,
        60: 140000000,
        120: 130000000,
    }
    threshold = threshold_by_window.get(config.tail_minutes, 140000000)
    sql = f"""
        WITH minute_feature AS (
            SELECT
                date,
                date::DATE::DATETIME AS trading_date,
                instrument,
                time,
                open,
                high,
                low,
                close,
                volume,
                amount,
                {imbalance} AS imbalance,
                ({bid}) AS bid_depth,
                ({ask}) AS ask_depth,
                (
                    (
                        ask_price1 * bid_volume1 + bid_price1 * ask_volume1
                    ) / (bid_volume1 + ask_volume1 + 1e-12)
                    - (ask_price1 + bid_price1) / 2.0
                ) / (ask_price1 - bid_price1 + 1e-12) AS micro_gap
            FROM {source}
        )
        SELECT
            trading_date AS date,
            instrument,
            AVG(CASE WHEN time >= {threshold} THEN imbalance ELSE NULL END)
                AS pressure_close,
            AVG(CASE WHEN time >= {threshold} THEN
                CASE WHEN imbalance >= 0 THEN 1.0 ELSE -1.0 END
                ELSE NULL END) AS signed_persistence,
            AVG(imbalance) AS pressure_day,
            AVG(CASE WHEN time >= {threshold} THEN micro_gap ELSE NULL END)
                AS micro_gap_close,
            FIRST(open) AS open_first,
            LAST(close) AS close_last,
            MAX(high) AS high_max,
            MIN(low) AS low_min,
            LAST(volume) AS volume_last,
            LAST(amount) AS amount_last,
            AVG(CASE WHEN time >= {threshold} THEN bid_depth - ask_depth ELSE NULL END)
                / (AVG(CASE WHEN time >= {threshold} THEN bid_depth + ask_depth ELSE NULL END)
                + 1e-12) AS tail_depth_imbalance
        FROM minute_feature
        GROUP BY trading_date, instrument
        ORDER BY date, instrument
    """
    start, end = _as_timestamp_bounds(start_date, end_date)
    try:
        result = dai.query(
            sql,
            filters={
                "date": [
                    start.strftime("%Y-%m-%d 00:00:00"),
                    (end + pd.Timedelta(days=1)).strftime("%Y-%m-%d 00:00:00"),
                ]
            },
            compression=True,
        ).df()
    except Exception:
        return pd.DataFrame()
    if result.empty:
        return result
    result["date"] = pd.to_datetime(result["date"], errors="coerce").dt.normalize()
    result["instrument"] = result["instrument"].astype(str)
    result["replenishment_asymmetry"] = (
        _numeric(result, "tail_depth_imbalance", 0.0)
        - _numeric(result, "pressure_day", 0.0)
    )
    signed_price_flow = (
        np.sign(_safe_divide(_numeric(result, "close_last"), _numeric(result, "open_first")) - 1.0)
        * _numeric(result, "pressure_close", 0.0).abs()
    )
    result["flow_confirmation"] = (
        0.7 * signed_price_flow
        + 0.3 * _numeric(result, "replenishment_asymmetry", 0.0)
    )
    return result


def _raw_bar_fields(depth: int) -> list[str]:
    fields = [
        "date",
        "instrument",
        "time",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "amount",
        "bid_price1",
        "ask_price1",
    ]
    for level in range(1, depth + 1):
        fields.extend([f"bid_volume{level}", f"ask_volume{level}"])
    return list(dict.fromkeys(fields))


def prepare_minute_features(
    bar: pd.DataFrame,
    config: HFFeatureConfig = DEFAULT_HF_CONFIG,
) -> pd.DataFrame:
    """Create normalized minute features from cumulative snapshot fields."""

    if bar.empty or not {"date", "instrument"}.issubset(bar.columns):
        return pd.DataFrame()
    frame = bar.copy()
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
    frame["instrument"] = frame["instrument"].astype(str)
    frame["day"] = frame["date"].dt.normalize()
    frame = frame.dropna(subset=["date", "instrument"]).sort_values(
        ["instrument", "day", "date"]
    )

    for name in ("open", "high", "low", "close", "volume", "amount", "bid_price1", "ask_price1"):
        frame[name] = _numeric(frame, name)
    frame["volume_increment"] = cumulative_to_increment(
        frame["volume"], frame["instrument"], frame["day"]
    )
    frame["amount_increment"] = cumulative_to_increment(
        frame["amount"], frame["instrument"], frame["day"]
    )

    weighted_bid = pd.Series(0.0, index=frame.index)
    weighted_ask = pd.Series(0.0, index=frame.index)
    for level in range(1, config.depth + 1):
        weight = 1.0 / level
        weighted_bid += weight * _numeric(frame, f"bid_volume{level}", 0.0).fillna(0.0)
        weighted_ask += weight * _numeric(frame, f"ask_volume{level}", 0.0).fillna(0.0)
    total_depth = weighted_bid + weighted_ask
    frame["bid_depth"] = weighted_bid
    frame["ask_depth"] = weighted_ask
    frame["imbalance"] = _safe_divide(weighted_bid - weighted_ask, total_depth).clip(-1, 1)

    valid_quote = (
        frame["bid_price1"].gt(0)
        & frame["ask_price1"].gt(0)
        & frame["ask_price1"].ge(frame["bid_price1"])
    )
    quote_mid = (frame["bid_price1"] + frame["ask_price1"]) / 2.0
    frame["mid"] = quote_mid.where(valid_quote, frame["close"])
    spread = (frame["ask_price1"] - frame["bid_price1"]).where(valid_quote)
    microprice = _safe_divide(
        frame["ask_price1"] * _numeric(frame, "bid_volume1", 0.0)
        + frame["bid_price1"] * _numeric(frame, "ask_volume1", 0.0),
        _numeric(frame, "bid_volume1", 0.0) + _numeric(frame, "ask_volume1", 0.0),
    )
    frame["micro_gap"] = _safe_divide(microprice - frame["mid"], spread).clip(-2, 2)

    keys = [frame["instrument"], frame["day"]]
    frame["mid_return"] = frame["mid"].groupby(keys, sort=False).pct_change()
    previous_return = frame["mid_return"].groupby(keys, sort=False).shift(1)
    bid_change = _safe_divide(
        frame["bid_depth"].groupby(keys, sort=False).diff(),
        frame["bid_depth"].groupby(keys, sort=False).shift(1).abs(),
    ).clip(-5, 5)
    ask_change = _safe_divide(
        frame["ask_depth"].groupby(keys, sort=False).diff(),
        frame["ask_depth"].groupby(keys, sort=False).shift(1).abs(),
    ).clip(-5, 5)
    frame["replenishment_asymmetry"] = np.where(
        previous_return.lt(0),
        bid_change,
        np.where(previous_return.gt(0), -ask_change, 0.0),
    )
    signed_volume = np.sign(frame["mid_return"].fillna(0.0)) * frame["volume_increment"]
    frame["signed_volume"] = signed_volume
    return frame


def aggregate_daily_hf(
    minute: pd.DataFrame,
    config: HFFeatureConfig = DEFAULT_HF_CONFIG,
) -> pd.DataFrame:
    if minute.empty:
        return pd.DataFrame()

    rows: list[dict[str, object]] = []
    for (day, instrument), group in minute.groupby(["day", "instrument"], sort=False):
        group = group.sort_values("date")
        tail = group.tail(config.tail_minutes)
        pressure = tail["imbalance"].mean()
        if not np.isfinite(pressure):
            pressure = np.nan
        sign = np.sign(pressure) if np.isfinite(pressure) else 0.0
        persistence = (
            (np.sign(tail["imbalance"].fillna(0.0)) == sign).mean() if sign else 0.0
        )
        volume_total = group["volume_increment"].sum()
        amount_total = group["amount_increment"].sum()
        vwap = amount_total / volume_total if volume_total > EPS else np.nan
        close_last = group["close"].dropna().iloc[-1] if group["close"].notna().any() else np.nan
        open_first = group["open"].dropna().iloc[0] if group["open"].notna().any() else np.nan
        realized_vol = float(
            np.sqrt(np.square(group["mid_return"].replace([np.inf, -np.inf], np.nan)).sum())
        )
        response = (
            abs(close_last / vwap - 1.0) / max(realized_vol, 1e-6)
            if np.isfinite(close_last) and np.isfinite(vwap) and vwap > EPS
            else np.nan
        )
        signed_flow = (
            group["signed_volume"].sum() / volume_total if volume_total > EPS else np.nan
        )
        pressure_day = group["imbalance"].mean()
        pressure_shift = pressure - pressure_day
        rows.append(
            {
                "date": pd.Timestamp(day),
                "instrument": str(instrument),
                "pressure_close": pressure,
                "signed_persistence": sign * persistence,
                "pressure_day": pressure_day,
                "micro_gap_close": tail["micro_gap"].mean(),
                "replenishment_asymmetry": tail["replenishment_asymmetry"].mean(),
                "flow_confirmation": 0.7 * signed_flow + 0.3 * pressure_shift,
                "open_first": open_first,
                "close_last": close_last,
                "high_max": group["high"].max(),
                "low_min": group["low"].min(),
                "volume_last": group["volume"].max(),
                "amount_last": group["amount"].max(),
                "realized_vol": realized_vol,
                "price_response": response,
            }
        )
    return pd.DataFrame(rows).sort_values(["date", "instrument"]).reset_index(drop=True)


def load_daily_hf_features(
    datasources: object,
    start_date: object,
    end_date: object,
    config: HFFeatureConfig = DEFAULT_HF_CONFIG,
) -> pd.DataFrame:
    """Load daily HF features, preferring DAI-side aggregation."""

    daily = _try_query_daily_hf_dai(datasources, start_date, end_date, config)
    if not daily.empty:
        open_first = _numeric(daily, "open_first")
        close_last = _numeric(daily, "close_last")
        high_max = _numeric(daily, "high_max")
        low_min = _numeric(daily, "low_min")
        vwap = _safe_divide(_numeric(daily, "amount_last"), _numeric(daily, "volume_last"))
        range_proxy = _safe_divide(high_max - low_min, open_first).abs().clip(lower=1e-6)
        daily["realized_vol"] = range_proxy
        daily["price_response"] = _safe_divide(
            _safe_divide(close_last, vwap).sub(1.0).abs(),
            range_proxy,
        ).clip(0, 20)
        return daily

    start, end = _as_timestamp_bounds(start_date, end_date)
    pieces: list[pd.DataFrame] = []
    cursor = start
    while cursor <= end:
        chunk_end = min(cursor + pd.Timedelta(days=19), end)
        raw = query_table(
            datasources,
            BAR_ALIASES,
            "bigalpha_2026_stock_bar1m",
            _raw_bar_fields(config.depth),
            cursor,
            chunk_end + pd.Timedelta(days=1),
        )
        if not raw.empty:
            pieces.append(aggregate_daily_hf(prepare_minute_features(raw, config), config))
        cursor = chunk_end + pd.Timedelta(days=1)
    if not pieces:
        return pd.DataFrame()
    return (
        pd.concat(pieces, ignore_index=True)
        .drop_duplicates(["date", "instrument"], keep="last")
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
    )


def attach_to_universe(
    universe: pd.DataFrame,
    daily: pd.DataFrame,
    raw_factor: pd.Series,
) -> pd.DataFrame:
    """Create the exact competition schema with neutral fallback values."""

    if universe.empty:
        return pd.DataFrame(columns=["date", "instrument", "factor"])
    factor_frame = daily[["date", "instrument"]].copy()
    factor_frame["raw_factor"] = pd.to_numeric(raw_factor, errors="coerce").replace(
        [np.inf, -np.inf], np.nan
    )
    factor_frame["factor"] = cross_section_rank(
        factor_frame, factor_frame["raw_factor"], fill_neutral=False
    )
    merged = universe.merge(
        factor_frame[["date", "instrument", "factor"]],
        on=["date", "instrument"],
        how="left",
    )
    # A neutral value preserves interface coverage without fabricating direction.
    merged["factor"] = pd.to_numeric(merged["factor"], errors="coerce").fillna(0.0)
    return validate_factor_output(merged[["date", "instrument", "factor"]])


def validate_factor_output(frame: pd.DataFrame) -> pd.DataFrame:
    required = ["date", "instrument", "factor"]
    if list(frame.columns) != required:
        raise ValueError(f"factor output columns must be exactly {required}")
    result = frame.copy()
    result["date"] = pd.to_datetime(result["date"], errors="coerce").dt.normalize()
    result["instrument"] = result["instrument"].astype(str)
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce")
    if result[["date", "instrument"]].isna().any().any():
        raise ValueError("date and instrument must not be missing")
    if result.duplicated(["date", "instrument"]).any():
        raise ValueError("duplicate date/instrument rows in factor output")
    if not np.isfinite(result["factor"]).all():
        raise ValueError("factor values must be finite")
    return result.sort_values(["date", "instrument"]).reset_index(drop=True)


def load_financial_history(
    datasources: object,
    start_date: object,
    end_date: object,
    lookback_days: int = 550,
) -> pd.DataFrame:
    start, end = _as_timestamp_bounds(start_date, end_date)
    fields = (
        "date",
        "instrument",
        "category",
        "shift",
        "report_date",
        "net_cffoa",
        "net_profit",
        "total_assets",
        "cash_received_from_sales_and_services",
        "operating_revenue",
        "total_operating_revenue",
    )
    return query_table(
        datasources,
        FINANCIAL_ALIASES,
        "bigalpha_2026_financial",
        fields,
        start - pd.Timedelta(days=lookback_days),
        end + pd.Timedelta(days=1),
    )


def attach_pit_quality(
    panel: pd.DataFrame,
    financial: pd.DataFrame,
) -> pd.DataFrame:
    """Attach latest disclosed financial state without forward-looking joins."""

    result = panel.sort_values(["instrument", "date"]).copy()
    if financial.empty or not {"date", "instrument"}.issubset(financial.columns):
        for name in ("quality_accrual", "quality_cash", "quality_change", "report_age"):
            result[name] = np.nan
        return result

    fin = financial.copy()
    fin["date"] = pd.to_datetime(fin["date"], errors="coerce").dt.normalize()
    fin["instrument"] = fin["instrument"].astype(str)
    if "report_date" in fin.columns:
        fin["report_date"] = pd.to_datetime(
            fin["report_date"], errors="coerce"
        ).dt.normalize()

    # Official schema provides category=ttm/mrq/lf and shift=report offset.
    # Use TTM for flow items and LF for the balance-sheet denominator.
    if "category" in fin.columns and fin["category"].notna().any():
        fin["category"] = fin["category"].astype(str).str.lower()
        if "shift" in fin.columns:
            shift = pd.to_numeric(fin["shift"], errors="coerce")
            fin = fin.loc[shift.fillna(0).eq(0)].copy()
        flow = fin.loc[fin["category"].eq("ttm")].copy()
        assets_frame = fin.loc[
            fin["category"].eq("lf"),
            [
                column
                for column in ("date", "instrument", "report_date", "total_assets")
                if column in fin.columns
            ],
        ].copy()
        merge_keys = ["date", "instrument"]
        if (
            "report_date" in flow.columns
            and "report_date" in assets_frame.columns
            and flow["report_date"].notna().any()
        ):
            merge_keys.append("report_date")
        assets_frame = assets_frame.drop_duplicates(merge_keys, keep="last").rename(
            columns={"total_assets": "total_assets_lf"}
        )
        fin = flow.merge(assets_frame, on=merge_keys, how="left")
        if "total_assets_lf" in fin.columns:
            fin["total_assets"] = fin["total_assets_lf"].combine_first(
                _numeric(fin, "total_assets")
            )

    assets = _numeric(fin, "total_assets").abs().where(lambda value: value > EPS)
    revenue = _numeric(fin, "operating_revenue")
    if revenue.isna().all():
        revenue = _numeric(fin, "total_operating_revenue")
    revenue = revenue.abs().where(lambda value: value > EPS)
    fin["quality_accrual"] = _safe_divide(
        _numeric(fin, "net_cffoa") - _numeric(fin, "net_profit"),
        assets,
    ).clip(-10, 10)
    fin["quality_cash"] = _safe_divide(
        _numeric(fin, "cash_received_from_sales_and_services"),
        revenue,
    ).clip(-10, 10)
    fin = fin.dropna(subset=["date", "instrument"]).sort_values(["instrument", "date"])
    fin["quality_level"] = 0.6 * fin["quality_accrual"] + 0.4 * fin["quality_cash"]
    fin["quality_change"] = fin.groupby("instrument", sort=False)["quality_level"].diff()
    # ``date`` is the official announcement date and is the PIT join key.
    fin["source_report_date"] = (
        pd.to_datetime(fin["report_date"], errors="coerce")
        if "report_date" in fin.columns
        else fin["date"]
    )
    fin["announcement_date"] = fin["date"]

    columns = [
        "date",
        "instrument",
        "source_report_date",
        "announcement_date",
        "quality_accrual",
        "quality_cash",
        "quality_change",
    ]
    merged_pieces: list[pd.DataFrame] = []
    for instrument, left in result.groupby("instrument", sort=False):
        right = fin.loc[fin["instrument"].eq(instrument), columns].sort_values("date")
        if right.empty:
            piece = left.copy()
            for name in columns[2:]:
                piece[name] = np.nan
        else:
            piece = pd.merge_asof(
                left.sort_values("date"),
                right,
                on="date",
                by="instrument",
                direction="backward",
                allow_exact_matches=True,
            )
        merged_pieces.append(piece)
    merged = pd.concat(merged_pieces, ignore_index=True)
    merged["report_age"] = (
        merged["date"] - pd.to_datetime(merged["announcement_date"], errors="coerce")
    ).dt.days
    return merged.sort_values(["date", "instrument"]).reset_index(drop=True)


"""Factor 1: persistent order-book pressure with price underreaction."""


import numpy as np
import pandas as pd



FACTOR_ID = "hf_pressure_underreaction_v1"
AI_MECHANISM = {
    "family": "PERSISTENT_BOOK_PRESSURE_UNDERREACTION",
    "selected_depth": 5,
    "selected_tail_minutes": 60,
    "allowed_windows": [15, 30, 60, 120],
    "allowed_depths": [1, 5, 10],
    "economic_hypothesis": (
        "Persistent closing book pressure predicts next-period returns when "
        "the contemporaneous price response remains incomplete."
    ),
}


def compute_hf_raw(
    daily: pd.DataFrame,
    config: HFFeatureConfig = DEFAULT_HF_CONFIG,
) -> pd.Series:
    """Compute the deterministic formula selected by constrained search."""

    pressure = pd.to_numeric(daily.get("pressure_close"), errors="coerce")
    persistence = pd.to_numeric(daily.get("signed_persistence"), errors="coerce").abs()
    response = (
        pd.to_numeric(daily.get("price_response"), errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .clip(lower=0.0, upper=20.0)
    )
    replenishment = (
        pd.to_numeric(daily.get("replenishment_asymmetry"), errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .clip(-5.0, 5.0)
    )
    micro_gap = (
        pd.to_numeric(daily.get("micro_gap_close"), errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .clip(-2.0, 2.0)
    )

    response_fill = response.median()
    if not np.isfinite(response_fill):
        response_fill = 0.0
    underreaction = pressure * persistence / (1.0 + response.fillna(response_fill))
    components = pd.concat(
        [
            underreaction.rename("underreaction"),
            (config.replenishment_weight * replenishment).rename("replenishment"),
            (config.microprice_weight * micro_gap).rename("microprice"),
        ],
        axis=1,
    )
    # Degrade by available component; only an entirely missing row stays missing.
    return components.sum(axis=1, min_count=1)


def main(datasources: object, start_date: object, end_date: object) -> pd.DataFrame:
    """BigAlpha submission entrypoint; returns exactly one daily factor."""

    universe = load_universe(datasources, start_date, end_date)
    if universe.empty:
        return pd.DataFrame(columns=["date", "instrument", "factor"])

    daily = load_daily_hf_features(
        datasources,
        start_date,
        end_date,
        DEFAULT_HF_CONFIG,
    )
    if daily.empty:
        neutral = universe.copy()
        neutral["factor"] = 0.0
        return validate_factor_output(neutral[["date", "instrument", "factor"]])
    raw = compute_hf_raw(daily, DEFAULT_HF_CONFIG)
    return attach_to_universe(universe, daily, raw)


"""Factor 2: PIT cash-flow quality confirmed by intraday resilience."""


from dataclasses import dataclass

import numpy as np
import pandas as pd



FACTOR_ID = "quality_flow_interaction_v1"
AI_MECHANISM = {
    "family": "PIT_QUALITY_FLOW_CONFIRMATION",
    "quality_weights": {"accrual": 0.6, "sales_cash": 0.4},
    "quality_change_weight": 0.2,
    "freshness_half_life_proxy_days": 120,
    "economic_hypothesis": (
        "Cash-flow quality is more informative at a daily horizon when adverse "
        "price shocks are absorbed and order flow confirms the fundamental direction."
    ),
}


@dataclass(frozen=True)
class InteractionConfig:
    quality_change_weight: float = 0.20
    freshness_days: int = 120
    resilience_weight: float = 0.25


DEFAULT_INTERACTION_CONFIG = InteractionConfig()


def compute_interaction_raw(
    panel: pd.DataFrame,
    config: InteractionConfig = DEFAULT_INTERACTION_CONFIG,
) -> pd.Series:
    """Combine PIT quality, directional confirmation and book resilience."""

    accrual_rank = cross_section_rank(
        panel,
        pd.to_numeric(panel.get("quality_accrual"), errors="coerce"),
        fill_neutral=True,
    )
    cash_rank = cross_section_rank(
        panel,
        pd.to_numeric(panel.get("quality_cash"), errors="coerce"),
        fill_neutral=True,
    )
    change_rank = cross_section_rank(
        panel,
        pd.to_numeric(panel.get("quality_change"), errors="coerce"),
        fill_neutral=True,
    )
    quality_level = 0.6 * accrual_rank + 0.4 * cash_rank
    quality_state = (
        (1.0 - config.quality_change_weight) * quality_level
        + config.quality_change_weight * change_rank
    )

    flow_rank = cross_section_rank(
        panel,
        pd.to_numeric(panel.get("flow_confirmation"), errors="coerce"),
        fill_neutral=True,
    )
    resilience_rank = cross_section_rank(
        panel,
        pd.to_numeric(panel.get("replenishment_asymmetry"), errors="coerce"),
        fill_neutral=True,
    )
    age = (
        pd.to_numeric(panel.get("report_age"), errors="coerce")
        .clip(lower=0.0, upper=720.0)
        .fillna(720.0)
    )
    freshness = 0.35 + 0.65 * np.exp(-age / float(config.freshness_days))

    aligned_confirmation = np.maximum(np.sign(quality_state) * flow_rank, 0.0)
    return (
        quality_state * aligned_confirmation * freshness
        + config.resilience_weight * quality_state * resilience_rank
    )


def main(datasources: object, start_date: object, end_date: object) -> pd.DataFrame:
    """BigAlpha submission entrypoint; returns exactly one daily factor."""

    universe = load_universe(datasources, start_date, end_date)
    if universe.empty:
        return pd.DataFrame(columns=["date", "instrument", "factor"])

    daily = load_daily_hf_features(
        datasources,
        start_date,
        end_date,
        DEFAULT_HF_CONFIG,
    )
    if daily.empty:
        neutral = universe.copy()
        neutral["factor"] = 0.0
        return validate_factor_output(neutral[["date", "instrument", "factor"]])

    panel = universe.merge(daily, on=["date", "instrument"], how="left")
    financial = load_financial_history(datasources, start_date, end_date)
    panel = attach_pit_quality(panel, financial)
    raw = compute_interaction_raw(panel, DEFAULT_INTERACTION_CONFIG)
    return attach_to_universe(universe, panel, raw)


"""Local replicas of the disclosed BigAlpha evaluation components."""


from dataclasses import dataclass
from typing import Iterable, Sequence

import numpy as np
import pandas as pd


LABEL_COLUMNS = (
    "ret_close_to_close",
    "ret_next_open_to_close",
    "ret_close_to_next_open",
)


def daily_prices_from_bar(bar: pd.DataFrame) -> pd.DataFrame:
    """Aggregate minute bars to the daily prices needed by the three labels."""

    frame = bar.copy()
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
    frame["day"] = frame["date"].dt.normalize()
    frame["instrument"] = frame["instrument"].astype(str)
    frame = frame.sort_values(["instrument", "date"])
    return (
        frame.groupby(["day", "instrument"], sort=False)
        .agg(open=("open", "first"), close=("close", "last"))
        .reset_index()
        .rename(columns={"day": "date"})
        .sort_values(["instrument", "date"])
        .reset_index(drop=True)
    )


def build_return_labels(daily_prices: pd.DataFrame) -> pd.DataFrame:
    """Build all plausible next-period labels disclosed only at a high level."""

    frame = daily_prices.copy().sort_values(["instrument", "date"])
    group = frame.groupby("instrument", sort=False)
    next_open = group["open"].shift(-1)
    next_close = group["close"].shift(-1)
    frame["ret_close_to_close"] = next_close / frame["close"] - 1.0
    frame["ret_next_open_to_close"] = next_close / next_open - 1.0
    frame["ret_close_to_next_open"] = next_open / frame["close"] - 1.0
    return frame[["date", "instrument", *LABEL_COLUMNS]]


def _winsorize_series(values: pd.Series, lower: float, upper: float) -> pd.Series:
    valid = values.dropna()
    if valid.empty:
        return values
    lo, hi = valid.quantile([lower, upper])
    return values.clip(lo, hi)


def preprocess_factor(
    factor: pd.DataFrame,
    exposures: pd.DataFrame | None = None,
    lower_quantile: float = 0.01,
    upper_quantile: float = 0.99,
) -> pd.DataFrame:
    """Daily winsorization, z-score and optional BARRA-style neutralization."""

    frame = factor[["date", "instrument", "factor"]].copy()
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce").dt.normalize()
    frame["factor"] = pd.to_numeric(frame["factor"], errors="coerce")
    frame["factor"] = frame.groupby("date", sort=False)["factor"].transform(
        lambda values: _winsorize_series(values, lower_quantile, upper_quantile)
    )
    mean = frame.groupby("date", sort=False)["factor"].transform("mean")
    std = frame.groupby("date", sort=False)["factor"].transform("std").replace(0, np.nan)
    frame["factor"] = (frame["factor"] - mean) / std

    if exposures is None or exposures.empty:
        return frame
    exp = exposures.copy()
    exp["date"] = pd.to_datetime(exp["date"], errors="coerce").dt.normalize()
    frame = frame.merge(exp, on=["date", "instrument"], how="left")
    exposure_columns = [
        column
        for column in exp.columns
        if column not in {"date", "instrument"}
        and pd.api.types.is_numeric_dtype(exp[column])
    ]
    if not exposure_columns:
        return frame[["date", "instrument", "factor"]]

    residuals = pd.Series(np.nan, index=frame.index, dtype=float)
    for _, indices in frame.groupby("date", sort=False).groups.items():
        block = frame.loc[indices]
        valid = block["factor"].notna() & block[exposure_columns].notna().all(axis=1)
        if valid.sum() <= len(exposure_columns) + 1:
            residuals.loc[indices] = block["factor"]
            continue
        x = block.loc[valid, exposure_columns].to_numpy(dtype=float)
        x = np.column_stack([np.ones(len(x)), x])
        y = block.loc[valid, "factor"].to_numpy(dtype=float)
        beta, *_ = np.linalg.lstsq(x, y, rcond=None)
        residuals.loc[block.index[valid]] = y - x @ beta
    frame["factor"] = residuals
    return frame[["date", "instrument", "factor"]]


def cross_section_zscore(
    frame: pd.DataFrame,
    columns: Sequence[str],
) -> pd.DataFrame:
    result = frame.copy()
    for column in columns:
        values = pd.to_numeric(result[column], errors="coerce")
        mean = values.groupby(result["date"], sort=False).transform("mean")
        std = values.groupby(result["date"], sort=False).transform("std").replace(0, np.nan)
        result[column] = (values - mean) / std
    return result


def rank_ic_series(
    merged: pd.DataFrame,
    factor_column: str = "factor",
    label_column: str = "ret_close_to_close",
) -> pd.Series:
    def one_day(block: pd.DataFrame) -> float:
        valid = block[[factor_column, label_column]].dropna()
        if len(valid) < 5:
            return np.nan
        return valid[factor_column].rank().corr(valid[label_column].rank())

    return merged.groupby("date", sort=False).apply(one_day, include_groups=False)


def long_short_returns(
    merged: pd.DataFrame,
    factor_column: str = "factor",
    label_column: str = "ret_close_to_close",
    quantiles: int = 5,
) -> pd.Series:
    def one_day(block: pd.DataFrame) -> float:
        valid = block[[factor_column, label_column]].dropna()
        if len(valid) < quantiles * 2:
            return np.nan
        ranks = valid[factor_column].rank(pct=True, method="average")
        top = valid.loc[ranks > 1.0 - 1.0 / quantiles, label_column].mean()
        bottom = valid.loc[ranks <= 1.0 / quantiles, label_column].mean()
        return float(top - bottom)

    return merged.groupby("date", sort=False).apply(one_day, include_groups=False)


def _safe_ratio(mean: float, std: float) -> float:
    return float(mean / std) if np.isfinite(std) and std > 1e-12 else np.nan


def evaluate_single_factor(
    factor: pd.DataFrame,
    labels: pd.DataFrame,
    exposures: pd.DataFrame | None = None,
) -> dict[str, dict[str, float]]:
    """Return A-item proxies for all three reasonable return labels."""

    processed = preprocess_factor(factor, exposures)
    merged = processed.merge(labels, on=["date", "instrument"], how="inner")
    output: dict[str, dict[str, float]] = {}
    for label in LABEL_COLUMNS:
        ic = rank_ic_series(merged, label_column=label).dropna()
        long_short = long_short_returns(merged, label_column=label).dropna()
        market = merged.groupby("date", sort=False)[label].mean()
        market_vol = merged.groupby("date", sort=False)[label].std()
        high_vol_cutoff = market_vol.quantile(0.75) if not market_vol.empty else np.nan
        stress_dates = market_vol.index[market_vol >= high_vol_cutoff]
        stress_ic = ic.reindex(stress_dates).dropna()
        output[label] = {
            "rank_ic_mean": float(ic.mean()) if not ic.empty else np.nan,
            "rank_ic_ir": _safe_ratio(float(ic.mean()), float(ic.std())),
            "long_short_sharpe": (
                _safe_ratio(float(long_short.mean()), float(long_short.std())) * np.sqrt(252)
                if not long_short.empty
                else np.nan
            ),
            "stress_ic_ir": _safe_ratio(
                float(stress_ic.mean()),
                float(stress_ic.std()),
            ),
            "up_market_ic": float(ic.reindex(market.index[market > 0]).mean()),
            "down_market_ic": float(ic.reindex(market.index[market <= 0]).mean()),
            "observations": float(len(ic)),
        }
    return output


@dataclass(frozen=True)
class ElasticNetConfig:
    window_days: int = 60
    step_days: int = 20
    alpha: float = 0.001
    l1_ratio: float = 0.5
    coefficient_epsilon: float = 1e-10


def rolling_elastic_net_scores(
    factor_panel: pd.DataFrame,
    target: pd.DataFrame,
    factor_columns: Sequence[str],
    target_column: str = "ret_close_to_close",
    config: ElasticNetConfig = ElasticNetConfig(),
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Approximate the disclosed 60-day/20-day Elastic Net ModelScore."""

    try:
        from sklearn.linear_model import ElasticNet
    except ImportError as exc:  # pragma: no cover
        raise RuntimeError("scikit-learn is required for Elastic Net evaluation") from exc

    merged = factor_panel.merge(
        target[["date", "instrument", target_column]],
        on=["date", "instrument"],
        how="inner",
    )
    merged = cross_section_zscore(merged, [*factor_columns, target_column])
    dates = np.array(sorted(pd.to_datetime(merged["date"].dropna().unique())))
    rows: list[dict[str, object]] = []
    for end_index in range(config.window_days, len(dates) + 1, config.step_days):
        window = dates[end_index - config.window_days : end_index]
        train = merged.loc[merged["date"].isin(window)].dropna(
            subset=[*factor_columns, target_column]
        )
        if len(train) <= len(factor_columns) + 2:
            continue
        model = ElasticNet(
            alpha=config.alpha,
            l1_ratio=config.l1_ratio,
            fit_intercept=True,
            max_iter=10000,
            random_state=0,
        )
        model.fit(
            train.loc[:, factor_columns].to_numpy(dtype=float),
            train[target_column].to_numpy(dtype=float),
        )
        row: dict[str, object] = {
            "window_start": pd.Timestamp(window[0]),
            "window_end": pd.Timestamp(window[-1]),
        }
        row.update(dict(zip(factor_columns, model.coef_, strict=True)))
        rows.append(row)

    weights = pd.DataFrame(rows)
    scores: list[dict[str, object]] = []
    for column in factor_columns:
        coefficients = (
            weights[column].abs()
            if column in weights.columns
            else pd.Series(dtype=float)
        )
        selected = coefficients > config.coefficient_epsilon
        mean_abs = float(coefficients.mean()) if not coefficients.empty else 0.0
        std_abs = float(coefficients.std(ddof=0)) if not coefficients.empty else 0.0
        score = mean_abs / (std_abs + 1e-12) if selected.any() else 0.0
        scores.append(
            {
                "factor": column,
                "model_score": score,
                "mean_abs_weight": mean_abs,
                "std_abs_weight": std_abs,
                "nonzero_window_ratio": float(selected.mean()) if len(selected) else 0.0,
            }
        )
    score_frame = pd.DataFrame(scores)
    if not score_frame.empty:
        score_frame["model_score_percentile"] = score_frame["model_score"].rank(pct=True)
    return score_frame, weights


def factor_rank_correlation(
    factor_panel: pd.DataFrame,
    factor_columns: Iterable[str],
) -> pd.DataFrame:
    ranked = factor_panel.copy()
    columns = list(factor_columns)
    for column in columns:
        ranked[column] = ranked.groupby("date", sort=False)[column].rank(pct=True)
    return ranked[columns].corr(method="pearson")


def chronological_gate(
    metrics_by_period: dict[str, dict[str, float]],
    development_period: str = "2019_2022",
    holdout_period: str = "2024",
) -> tuple[bool, list[str]]:
    """Apply the frozen factor admission rules from the implementation plan."""

    reasons: list[str] = []
    ic_values = {
        period: values.get("rank_ic_mean", np.nan)
        for period, values in metrics_by_period.items()
    }
    finite = {period: value for period, value in ic_values.items() if np.isfinite(value)}
    signs = {int(np.sign(value)) for value in finite.values() if abs(value) > 1e-12}
    if len(finite) != len(metrics_by_period) or len(signs) != 1:
        reasons.append("Rank IC direction is not stable across all periods")
    development = abs(ic_values.get(development_period, np.nan))
    holdout = abs(ic_values.get(holdout_period, np.nan))
    if not np.isfinite(development) or not np.isfinite(holdout) or holdout < 0.5 * development:
        reasons.append("2024 Rank IC is below 50% of the development-period magnitude")
    return not reasons, reasons


"""Constrained, auditable candidate enumeration for the AI track."""


from dataclasses import asdict, dataclass
from itertools import product
from typing import Callable, Iterable

import pandas as pd


@dataclass(frozen=True)
class CandidateSpec:
    candidate_id: str
    family: str
    tail_minutes: int
    depth: int
    replenishment_weight: float
    microprice_weight: float
    freshness_days: int
    quality_change_weight: float

    def to_dict(self) -> dict[str, object]:
        return asdict(self)


def enumerate_candidate_specs() -> list[CandidateSpec]:
    """Return 48 candidates: 24 per economic mechanism family."""

    specs: list[CandidateSpec] = []
    index = 1
    for tail, depth, replenish in product(
        (15, 30, 60, 120),
        (1, 5, 10),
        (0.20, 0.35),
    ):
        specs.append(
            CandidateSpec(
                candidate_id=f"hf_{index:02d}",
                family="PERSISTENT_BOOK_PRESSURE_UNDERREACTION",
                tail_minutes=tail,
                depth=depth,
                replenishment_weight=replenish,
                microprice_weight=0.20,
                freshness_days=120,
                quality_change_weight=0.20,
            )
        )
        index += 1

    index = 1
    for tail, depth, freshness in product(
        (15, 30, 60, 120),
        (1, 5, 10),
        (90, 180),
    ):
        specs.append(
            CandidateSpec(
                candidate_id=f"interaction_{index:02d}",
                family="PIT_QUALITY_FLOW_CONFIRMATION",
                tail_minutes=tail,
                depth=depth,
                replenishment_weight=0.25,
                microprice_weight=0.0,
                freshness_days=freshness,
                quality_change_weight=0.20,
            )
        )
        index += 1
    return specs


def run_constrained_search(
    evaluator: Callable[[CandidateSpec], dict[str, float]],
    specs: Iterable[CandidateSpec] | None = None,
) -> pd.DataFrame:
    """Evaluate candidates through a caller-provided, platform-local evaluator."""

    rows: list[dict[str, object]] = []
    for spec in specs or enumerate_candidate_specs():
        metrics = evaluator(spec)
        row = spec.to_dict()
        row.update(metrics)
        rows.append(row)
    result = pd.DataFrame(rows)
    sort_columns = [
        column
        for column in ("admission_pass", "model_score", "rank_ic_ir")
        if column in result.columns
    ]
    if sort_columns:
        result = result.sort_values(sort_columns, ascending=False)
    return result.reset_index(drop=True)


"""Platform-local candidate construction and chronological selection workflow."""


from dataclasses import dataclass
from typing import Iterable, Sequence

import numpy as np
import pandas as pd



PERIODS = {
    "2019_2022": (pd.Timestamp("2019-01-01"), pd.Timestamp("2022-12-31")),
    "2023": (pd.Timestamp("2023-01-01"), pd.Timestamp("2023-12-31")),
    "2024": (pd.Timestamp("2024-01-01"), pd.Timestamp("2024-12-31")),
}


@dataclass
class SearchResult:
    candidate_library: pd.DataFrame
    summary: pd.DataFrame
    elastic_net_weights: pd.DataFrame
    rank_correlation: pd.DataFrame


def daily_prices_from_hf_features(daily_features: pd.DataFrame) -> pd.DataFrame:
    """Convert the cached DAI daily aggregation into the label price schema."""

    required = {"date", "instrument", "open_first", "close_last"}
    missing = required.difference(daily_features.columns)
    if missing:
        raise ValueError(f"daily HF features are missing price columns: {sorted(missing)}")
    return daily_features[
        ["date", "instrument", "open_first", "close_last"]
    ].rename(columns={"open_first": "open", "close_last": "close"})


def build_candidate_library(
    datasources: object,
    start_date: object = "2019-01-01",
    end_date: object = "2024-12-31",
    specs: Iterable[CandidateSpec] | None = None,
) -> pd.DataFrame:
    """Materialize the 48 constrained candidates with reusable HF caches."""

    selected_specs = list(specs or enumerate_candidate_specs())
    universe = load_universe(datasources, start_date, end_date)
    if universe.empty:
        return pd.DataFrame(columns=["date", "instrument"])

    hf_cache: dict[tuple[int, int], pd.DataFrame] = {}
    financial = load_financial_history(datasources, start_date, end_date)
    library = universe.copy()
    for spec in selected_specs:
        cache_key = (spec.depth, spec.tail_minutes)
        if cache_key not in hf_cache:
            config = HFFeatureConfig(
                depth=spec.depth,
                tail_minutes=spec.tail_minutes,
                replenishment_weight=spec.replenishment_weight,
                microprice_weight=spec.microprice_weight,
            )
            hf_cache[cache_key] = load_daily_hf_features(
                datasources,
                start_date,
                end_date,
                config,
            )
        daily = hf_cache[cache_key]
        if daily.empty:
            library[spec.candidate_id] = 0.0
            continue

        if spec.family == "PERSISTENT_BOOK_PRESSURE_UNDERREACTION":
            factor_config = HFFeatureConfig(
                depth=spec.depth,
                tail_minutes=spec.tail_minutes,
                replenishment_weight=spec.replenishment_weight,
                microprice_weight=spec.microprice_weight,
            )
            raw = compute_hf_raw(daily, factor_config)
            factor = attach_to_universe(universe, daily, raw)
        elif spec.family == "PIT_QUALITY_FLOW_CONFIRMATION":
            panel = attach_pit_quality(
                universe.merge(daily, on=["date", "instrument"], how="left"),
                financial,
            )
            interaction_config = InteractionConfig(
                quality_change_weight=spec.quality_change_weight,
                freshness_days=spec.freshness_days,
                resilience_weight=0.25,
            )
            raw = compute_interaction_raw(panel, interaction_config)
            factor = attach_to_universe(universe, panel, raw)
        else:
            raise ValueError(f"unsupported candidate family: {spec.family}")
        library = library.merge(
            factor.rename(columns={"factor": spec.candidate_id}),
            on=["date", "instrument"],
            how="left",
        )
    return library.sort_values(["date", "instrument"]).reset_index(drop=True)


def _period_metrics(
    factor: pd.DataFrame,
    labels: pd.DataFrame,
    exposures: pd.DataFrame | None,
) -> dict[str, dict[str, float]]:
    metrics: dict[str, dict[str, float]] = {}
    for period, (start, end) in PERIODS.items():
        factor_slice = factor.loc[factor["date"].between(start, end)]
        label_slice = labels.loc[labels["date"].between(start, end)]
        exposure_slice = (
            exposures.loc[exposures["date"].between(start, end)]
            if exposures is not None and not exposures.empty
            else None
        )
        evaluated = evaluate_single_factor(factor_slice, label_slice, exposure_slice)
        metrics[period] = evaluated["ret_close_to_close"]
    return metrics


def evaluate_candidate_library(
    candidate_library: pd.DataFrame,
    daily_prices: pd.DataFrame,
    exposures: pd.DataFrame | None = None,
    factorlib: pd.DataFrame | None = None,
    specs: Sequence[CandidateSpec] | None = None,
) -> SearchResult:
    """Apply chronology, A proxies, Elastic Net and correlation admission gates."""

    selected_specs = list(specs or enumerate_candidate_specs())
    candidate_columns = [
        spec.candidate_id
        for spec in selected_specs
        if spec.candidate_id in candidate_library.columns
    ]
    labels = build_return_labels(daily_prices)
    processed = candidate_library[["date", "instrument"]].copy()
    summary_rows: list[dict[str, object]] = []
    for spec in selected_specs:
        if spec.candidate_id not in candidate_columns:
            continue
        factor = candidate_library[
            ["date", "instrument", spec.candidate_id]
        ].rename(columns={spec.candidate_id: "factor"})
        preprocessed = preprocess_factor(factor, exposures).rename(
            columns={"factor": spec.candidate_id}
        )
        processed = processed.merge(
            preprocessed,
            on=["date", "instrument"],
            how="left",
        )
        period_metrics = _period_metrics(factor, labels, exposures)
        row: dict[str, object] = spec.to_dict()
        for period, metrics in period_metrics.items():
            row[f"{period}_rank_ic_mean"] = metrics["rank_ic_mean"]
            row[f"{period}_rank_ic_ir"] = metrics["rank_ic_ir"]
            row[f"{period}_long_short_sharpe"] = metrics["long_short_sharpe"]
            row[f"{period}_stress_ic_ir"] = metrics["stress_ic_ir"]
        label_metrics = evaluate_single_factor(factor, labels, exposures)
        label_signs = {
            int(np.sign(values["rank_ic_mean"]))
            for values in label_metrics.values()
            if np.isfinite(values["rank_ic_mean"])
            and abs(values["rank_ic_mean"]) > 1e-12
        }
        row["three_label_direction_consistent"] = len(label_signs) == 1
        summary_rows.append(row)

    regression_panel = processed
    regression_columns = list(candidate_columns)
    factorlib_columns: list[str] = []
    if factorlib is not None and not factorlib.empty:
        factorlib_columns = [
            column
            for column in factorlib.columns
            if column not in {"date", "instrument"}
            and pd.api.types.is_numeric_dtype(factorlib[column])
        ]
        regression_panel = regression_panel.merge(
            factorlib[["date", "instrument", *factorlib_columns]],
            on=["date", "instrument"],
            how="left",
        )
        regression_columns.extend(factorlib_columns)

    scores, weights = rolling_elastic_net_scores(
        regression_panel,
        labels,
        regression_columns,
    )
    score_lookup = scores.set_index("factor") if not scores.empty else pd.DataFrame()
    correlation = factor_rank_correlation(regression_panel, regression_columns)
    summary = pd.DataFrame(summary_rows)
    for index, row in summary.iterrows():
        candidate = str(row["candidate_id"])
        if not score_lookup.empty and candidate in score_lookup.index:
            for column in (
                "model_score",
                "model_score_percentile",
                "nonzero_window_ratio",
            ):
                summary.loc[index, column] = score_lookup.loc[candidate, column]
        if factorlib_columns and candidate in correlation.index:
            summary.loc[index, "max_factorlib_rank_correlation"] = (
                correlation.loc[candidate, factorlib_columns].abs().max()
            )
        else:
            summary.loc[index, "max_factorlib_rank_correlation"] = np.nan

    dev = summary["2019_2022_rank_ic_mean"].abs()
    holdout = summary["2024_rank_ic_mean"].abs()
    period_signs = np.column_stack(
        [
            np.sign(summary["2019_2022_rank_ic_mean"]),
            np.sign(summary["2023_rank_ic_mean"]),
            np.sign(summary["2024_rank_ic_mean"]),
        ]
    )
    same_period_direction = np.all(period_signs == period_signs[:, [0]], axis=1)
    correlation_gate = (
        summary["max_factorlib_rank_correlation"].lt(0.7)
        | summary["max_factorlib_rank_correlation"].isna()
    )
    summary["admission_pass"] = (
        same_period_direction
        & summary["three_label_direction_consistent"].astype(bool)
        & holdout.ge(0.5 * dev)
        & summary["nonzero_window_ratio"].fillna(0.0).ge(0.6)
        & correlation_gate
    )
    summary = summary.sort_values(
        ["admission_pass", "model_score_percentile", "2024_rank_ic_ir"],
        ascending=False,
    ).reset_index(drop=True)
    return SearchResult(
        candidate_library=candidate_library,
        summary=summary,
        elastic_net_weights=weights,
        rank_correlation=correlation,
    )


def select_private_candidates(result: SearchResult) -> tuple[str, str]:
    """Select one candidate per family, enforcing the pairwise correlation target."""

    eligible = result.summary.loc[result.summary["admission_pass"]].copy()
    if eligible.empty:
        raise ValueError("no candidate passed all admission gates")
    hf = eligible.loc[
        eligible["family"].eq("PERSISTENT_BOOK_PRESSURE_UNDERREACTION")
    ]
    interaction = eligible.loc[eligible["family"].eq("PIT_QUALITY_FLOW_CONFIRMATION")]
    if hf.empty or interaction.empty:
        raise ValueError("both mechanism families must have an eligible candidate")
    for hf_id in hf["candidate_id"]:
        for interaction_id in interaction["candidate_id"]:
            corr = result.rank_correlation.loc[hf_id, interaction_id]
            if np.isfinite(corr) and abs(corr) < 0.5:
                return str(hf_id), str(interaction_id)
    raise ValueError("no eligible cross-family pair met the 0.5 correlation target")


## 运行入口

下面代码默认注释，确认字段和资源后逐段运行。结果只保存在平台内。

In [ ]:
# datasources = {}
# specs = enumerate_candidate_specs()
# candidate_library = build_candidate_library(
#     datasources, '2019-01-01', '2024-12-31', specs
# )
# daily_features = load_daily_hf_features(
#     datasources, '2019-01-01', '2024-12-31'
# )
# daily_prices = daily_prices_from_hf_features(daily_features)
# result = evaluate_candidate_library(
#     candidate_library, daily_prices,
#     exposures=None, factorlib=None, specs=specs
# )
# result.summary.head(20)